In [1]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-sonnet-4-5"

In [2]:
# Helper functions
from anthropic.types import Message


def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[], tools=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message


def text_from_message(message):
    return "\n".join([block.text for block in message.content if block.type == "text"])

In [3]:
web_search_schema = {
    "type": "web_search_20250305",
    "name": "web_search",
    "max_uses": 5
}

In [7]:
messages = []
add_user_message(
    messages,
    """
    What's the current stock price of NVIDIA?
    """,
)
response = chat(messages, tools=[web_search_schema])
response

Message(id='msg_01D1ZUkTF9f945jc6kpS4vb5', content=[ServerToolUseBlock(id='srvtoolu_01AtvNBPtUx9smobo9HT8vH3', input={'query': 'NVIDIA stock price today'}, name='web_search', type='server_tool_use'), WebSearchToolResultBlock(content=[WebSearchResultBlock(encrypted_content='EogOCioIDBgCIiRmNzNmMWNlNi0zMDAyLTRlZWYtOGMwZC00YTM0ZGFiNGYwZDcSDHt7soTVLdqcuFPQTBoM4rAvMq9lIgIlBX62IjAkpAtLKdAPqvoodKraimEWqR+gt4GOk/IrMRZUfecZuHb7Sp/os3IeY34eSoGQmNUqiw2U6Ch0LxmxiBKUf+vkklho6+eRTYlHd/N9C5UsMT5yeb+mw6aPk4MV84dQczp7doTTf+Qy8iO4FxJTliPZH6T7QCHuthwHwHbQFKVrkOLwuUs82nNJLO1Cp7pP9iu6mxCt3jIuiJCZqXguPpQW6b9iTY8P63QcbcGnMPkY8BhaxcSpDR/TZwG/FkWaU5Peyny+2IcHi82G9Y9l2mmZVlMqAZaOQ/3ZhpQTW6cobI+UmfBfTh3sAwxhOL234JB0RMZf7YaXQnGZu7upSrM+ScYszT8hg4Tn3F7DoHhyuFS4QeiQMd/0n/Vy6sZD4YafOh2QhKPVJHrBV+iPQcUZD+NJlEHtu7Mz25/WeEHU9q/aXAVcPKX+CP6EjEfexHh+qpVyVTRh5wcKktCq7sYspolqHGC+k0dYcWkx/d2obbv75GA5DchmCrqeen+eFt2D5NgHTMr2hW3XaaqeLXRj7cS4B5cPza/gYK3i7TLGfeKJaKocmSeFb2pPSqr4HctyKQZsqwZdzh45u+AAHo4CTylm2kVUpAPtpUudYljKrMS+Hf

In [8]:
for block in response.content:
    if block.type == "text":
        print(block.text)
    # Check citations if present
    if hasattr(block, 'citations') and block.citations:
        for citation in block.citations:
            print(f"Source: {citation.url}")

Based on the latest information available, 
NVIDIA (NVDA) stock is currently priced at $186.10
Source: https://robinhood.com/us/en/stocks/NVDA/
. 
On January 26, 2026, the stock traded between a low of $183.50 and a high of $189.60
Source: https://robinhood.com/us/en/stocks/NVDA/
.

The stock has a 
market capitalization of $4.62 trillion
Source: https://robinhood.com/us/en/stocks/NVDA/
 and 
trades at a price-to-earnings (P/E) ratio of 46.48 with a dividend yield of 2.1%
Source: https://robinhood.com/us/en/stocks/NVDA/
.
